A framework designed to compare performance of retrieved results based on word-level embeddings, sentence-level embeddings, and embeddings pre-trained along with graph metrics.

In [222]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from sentence_transformers import SentenceTransformer
import csv
import ast
from sklearn.metrics.pairwise import cosine_similarity

1. Vector Database Organization

In [43]:
INIT_WORD_VECTOR_DB = []

In [ ]:
BERT_TOKENIZER = BertTokenizer.from_pretrained('bert-base-uncased')
BERT_MODEL = BertModel.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 789.51it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [105]:
BERT_VECTOR_DB =  []

In [48]:
with open('./cat-facts.txt', encoding='UTF-8') as file:
    INIT_WORD_VECTOR_DB = file.readlines()

In [116]:
def bertSentToEmbedding(sent: str) -> Tensor:

    tokenizedLine = BERT_TOKENIZER.tokenize(sent)
    
    inputs = torch.tensor(BERT_TOKENIZER.convert_tokens_to_ids(tokenizedLine)).unsqueeze(0)
    
    with torch.no_grad():
    
        output = BERT_MODEL(inputs)
    
    embeddings = F.normalize(output.last_hidden_state[:, 0, :], p = 2, dim = 1)

    return embeddings

In [107]:
#Creating vector database based on BERT-model derived sentence-level embeddings.

for line in INIT_WORD_VECTOR_DB:

    embedding = bertSentToEmbedding(line)

    BERT_VECTOR_DB.append(embedding)

In [110]:
with open("BERT_VECTOR_DB_1.csv", 'w',  encoding = 'utf-8') as file:
    
    writer  = csv.writer(file)

    writer.writerow(["Sentences", "BERT Vectors"])

    for i in range(len(BERT_VECTOR_DB)):
        
        writer.writerow([INIT_WORD_VECTOR_DB[i], BERT_VECTOR_DB[i]])
        
        

1. BERT Embedding Based DB

In [245]:
def cleanStoredBertTensor(val: str):
    
    startingIndex = 0
    endingIndex = 0
    
    endingFound = False

    for i in range(len(val)):

        if val[i] == '[':
            startingIndex = i

    for i in range(len(val)):

        if val[i] == ']' and not endingFound:
            endingIndex = i
            endingFound = True

    return ast.literal_eval(val[startingIndex:endingIndex + 1])
    


In [111]:
BERT_VECTOR_DB = pd.read_csv("BERT_VECTOR_DB_1.csv")

In [112]:
BERT_VECTOR_DB 

,Sentences,BERT Vectors
0,"On average, cats spend 2/3 of every day sleepi...","tensor([[ 3.2025e-02, -1.9272e-03, 7.5102e-03..."
1,"Unlike dogs, cats do not have a sweet tooth. S...","tensor([[ 2.5943e-02, 1.3421e-02, -1.0437e-03..."
2,"When a cat chases its prey, it keeps its head ...","tensor([[ 4.9998e-02, 7.0072e-03, 1.5407e-02..."
3,The technical term for a cat’s hairball is a “...,"tensor([[ 2.2468e-03, 6.4672e-03, 2.8188e-02..."
4,A group of cats is called a “clowder.”\r\n,"tensor([[-9.7049e-04, 4.5340e-03, 3.1755e-02..."
...,...,...
145,"Like birds, cats have a homing ability that us...","tensor([[ 3.0573e-02, 1.5667e-02, 2.0914e-02..."
146,Cats bury their feces to cover their trails fr...,"tensor([[ 1.8749e-02, -9.4188e-03, 1.5741e-02..."
147,Cats sleep 16 to 18 hours per day. When cats a...,"tensor([[ 2.3224e-02, 1.4838e-02, 2.2194e-02..."
148,"Besides smelling with their nose, cats can sme...","tensor([[ 1.1026e-02, 5.3487e-02, 1.8455e-02..."


In [243]:
req = input("Please input your request about cats.")

reqEmbedding = bertSentToEmbedding(req)

In [246]:
greatestMatch = None 
greatestCosSim = -10

for row in BERT_VECTOR_DB.iterrows():
    
    dbVector = cleanStoredBertTensor(row[1]['BERT Vectors'])
    
    dbVector = torch.tensor(dbVector).reshape(1, -1)
    
    dbReqCos = cosine_similarity(dbVector, reqEmbedding)

    if dbReqCos > greatestCosSim:
        
        greatestMatch = row[1]['Sentences']
        greatestCosSim = dbReqCos
   

In [247]:
print("Query: {}\nResponse: {}".format(req, greatestMatch))

Query: When was the first cat show?
Response: Researchers believe the word “tabby” comes from Attabiyah, a neighborhood in Baghdad, Iraq. Tabbies got their name because their striped coats resembled the famous wavy patterns in the silk produced in this city.



2. S-BERT Embedding Based DB

In [186]:
SBERT = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6318.76it/s]


In [192]:
SBERT_VECTOR_DB = []

In [193]:
def sbertSentToEmbedding(sent: str) -> Tensor:

    embeddings = SBERT.encode(sent, normalize_embeddings = True)

    return embeddings

In [ ]:
for line in INIT_WORD_VECTOR_DB:

    embedding = sbertSentToEmbedding(line)

    SBERT_VECTOR_DB.append(embedding)
    

In [196]:
with open("SBERT_VECTOR_DB_1.csv", 'w',  encoding = 'utf-8') as file:
    
    writer  = csv.writer(file)

    writer.writerow(["Sentences", "SBERT Vectors"])

    for i in range(len(BERT_VECTOR_DB)):
        
        writer.writerow([INIT_WORD_VECTOR_DB[i], SBERT_VECTOR_DB[i]])
        

In [199]:
SBERT_VECTOR_DB = pd.read_csv("SBERT_VECTOR_DB_1.csv")

In [239]:
req = input("Please input your request about cats.")

reqEmbedding = sbertSentToEmbedding(req)

In [228]:
def cleanStoredSbertTensor(val: str):
    
    start = val.find('[')
    end = val.rfind(']')
    
    if start == -1 or end == -1:
        return "" # Handle empty or corrupt strings safely
        
    return val[start:end + 1]

# Parsing into a complete list
cleaned_str = cleanStoredSbertTensor(dbVector)
numerical_vector = [float(x) for x in cleaned_str.strip("[]").split()]

In [231]:
print(type(numerical_vector))

<class 'list'>


In [ ]:
greatestMatch = None 
greatestCosSim = -10

for row in SBERT_VECTOR_DB.iterrows():
    
    dbVector = cleanStoredSbertTensor(row[1]['SBERT Vectors'])
    
    numericalVector = [float(x) for x in dbVector.strip("[]").split()]

    numericalVector = torch.tensor(numericalVector).reshape(1, -1)
        
    dbReqCos = cosine_similarity(numericalVector, reqEmbedding.reshape(1, -1))
    
    if dbReqCos > greatestCosSim:
            
        greatestMatch = row[1]['Sentences']
        greatestCosSim = dbReqCos



In [242]:
print("Query: {}\nResponse: {}".format(req, greatestMatch))

Query: When was the first cat show?
Response: The first cat show was organized in 1871 in London. Cat shows later became a worldwide craze.

